In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from torch.utils.data import Dataset

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df   = pd.read_csv("../data/processed/val.csv")

# T5 only needs clickbait rows — it learns to rewrite them
train_cb = train_df[train_df["label"] == 1].reset_index(drop=True)
val_cb   = val_df[val_df["label"] == 1].reset_index(drop=True)

print(f"Clickbait rows — Train: {len(train_cb)} | Val: {len(val_cb)}")
print(train_cb["headline"].sample(3).tolist())

Clickbait rows — Train: 12810 | Val: 1601
["The strict but straightforward daily routine of the world's most successful, wealthy and stress-free people - including waking at 5am and taking at LEAST an hour for lunch", 'How To Eat Nothing But Pizza For Every Single Meal', 'Can You Tell How Old These Fucking Pennies Are']


In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model      = model.to(device)

print("Paraphrase model loaded on", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paraphrase model loaded on mps


In [4]:
import re

def clean_headline(text):
    # Remove common clickbait phrases
    patterns = [
        r"you won'?t believe",
        r"this is why",
        r"this is what",
        r"here'?s why",
        r"here'?s what",
        r"what happens (when|next|after)",
        r"the reason (is|why)",
        r"\d+ (things|reasons|ways|facts|tips|tricks|secrets|signs|habits)",
        r"\.\.\.+",
    ]
    cleaned = text
    for p in patterns:
        cleaned = re.sub(p, "", cleaned, flags=re.IGNORECASE)

    # Remove excessive punctuation and capitalisation
    cleaned = re.sub(r"[!]{1,}", ".", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = cleaned.strip("–—-:, ")

    # Sentence case
    cleaned = cleaned[0].upper() + cleaned[1:] if cleaned else text
    return cleaned

# Test it
sample = "You Won't Believe What This Student Did – Professors Are Shocked!"
print("Original:", sample)
print("Cleaned :", clean_headline(sample))

Original: You Won't Believe What This Student Did – Professors Are Shocked!
Cleaned : What This Student Did – Professors Are Shocked.


In [5]:
train_cb["target"] = train_cb["headline"].apply(clean_headline)
val_cb["target"]   = val_cb["headline"].apply(clean_headline)

# Preview a few pairs
for _, row in train_cb.sample(4).iterrows():
    print(f"IN  : {row['headline']}")
    print(f"OUT : {row['target']}")
    print()

IN  : 7 Essays To Read: When Schools Are Racist, "Depressiongrams," And #ShoutYourAbortion
OUT : 7 Essays To Read: When Schools Are Racist, "Depressiongrams," And #ShoutYourAbortion

IN  : Would You Pass Environmental Science Now
OUT : Would You Pass Environmental Science Now

IN  : Is this mysterious rumbling coming from Metro? Listen for yourself
OUT : Is this mysterious rumbling coming from Metro? Listen for yourself

IN  : 17 Impossibly Easy Kitchen DIYs That Only Look Expensive
OUT : 17 Impossibly Easy Kitchen DIYs That Only Look Expensive



In [6]:
class RewriteDataset(Dataset):
    def __init__(self, df, tokenizer, max_input=128, max_target=64):
        self.inputs  = ["rewrite clickbait: " + h for h in df["headline"]]
        self.targets = list(df["target"])
        self.tokenizer   = tokenizer
        self.max_input   = max_input
        self.max_target  = max_target

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        model_inputs = self.tokenizer(
            self.inputs[idx],
            max_length=self.max_input,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        labels = self.tokenizer(
            self.targets[idx],
            max_length=self.max_target,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        model_inputs = {k: v.squeeze() for k, v in model_inputs.items()}
        model_inputs["labels"] = labels["input_ids"].squeeze()
        return model_inputs

train_dataset = RewriteDataset(train_cb, tokenizer)
val_dataset   = RewriteDataset(val_cb,   tokenizer)

print(f"Datasets ready — Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Datasets ready — Train: 12810 | Val: 1601


In [8]:
training_args = TrainingArguments(
    output_dir                  = "../models/t5_rewriter",
    num_train_epochs            = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_dir                 = "../models/t5_rewriter/logs",
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
)

print("Training args OK")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training args OK


In [9]:
trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
)

trainer.train()

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.005414,0.002416
2,0.001928,0.001291
3,0.001388,0.001158


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4806, training_loss=0.1326387483478166, metrics={'train_runtime': 1004.8537, 'train_samples_per_second': 38.244, 'train_steps_per_second': 4.783, 'total_flos': 1300296357642240.0, 'train_loss': 0.1326387483478166, 'epoch': 3.0})

In [10]:
model.save_pretrained("../models/t5_rewriter")
tokenizer.save_pretrained("../models/t5_rewriter")
print("T5 model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

T5 model saved!


In [14]:
def rewrite(headline):
    input_text = f"paraphrase: {headline} </s>"
    inputs = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length     = 128,
        truncation     = True,
        padding        = "max_length"
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length           = 64,
            num_beams            = 5,
            repetition_penalty   = 2.5,
            no_repeat_ngram_size = 2,
            early_stopping       = True
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Original : {headline}")
    print(f"Rewritten: {result}")
    print()

rewrite("You Won't Believe What This Student Did With AI!")
rewrite("10 Shocking Reasons Why Your Diet Is Failing You")
rewrite("This Simple Trick Will Change Your Life Forever")

Original : You Won't Believe What This Student Did With AI!
Rewritten: You'd be amazed at the progress this student made using AI!

Original : 10 Shocking Reasons Why Your Diet Is Failing You
Rewritten: What are the 10 surprising reasons why your diet is not working?

Original : This Simple Trick Will Change Your Life Forever
Rewritten: A Quick Trick That Will Change Your Life Forever.



In [15]:
model.save_pretrained("../models/t5_rewriter")
tokenizer.save_pretrained("../models/t5_rewriter")
print("Paraphrase model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Paraphrase model saved!
